In [5]:
"""
HC-use decoding from aperiodic EEG — confound-controlled.

Answers "is the decode just age / nicotine / alcohol?" by comparing four LOSO models:
  1. covariates only         (age, daily nicotine, weekly alcohol)   -> should be ~chance
  2. EEG only                (exponent+offset × EC/EO × 64 ch)
  3. EEG + covariates
  4. EEG residualized on covariates  (confounds regressed OUT of every EEG feature,
     fold-wise inside CV) -> the clean test; + permutation p-value

Also saves the EEG-only coefficient topomap (which channels drive the decode).

Inputs
  derivatives/preproc/specparam/aperiodic_per_subject_channel.csv
  participants.tsv
  <phenotype>/lifestyle.tsv
Outputs (derivatives/preproc/ml/)
  hc_covariate_control_results.txt
  hc_coef_topomap.png

Requires: pandas numpy scikit-learn scipy matplotlib mne
"""

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

# ============================================================
# CONFIG   (point these at wherever your data actually lives)
# ============================================================
BIDS_ROOT     = "/Users/elizabethkaplan/Desktop/ds007615/ds007615"
DERIV_ROOT    = os.path.join(BIDS_ROOT, "derivatives", "preproc")
APERIODIC_CSV = os.path.join(DERIV_ROOT, "specparam", "aperiodic_per_subject_channel.csv")
PARTICIPANTS  = os.path.join(BIDS_ROOT, "participants.tsv")
PHENOTYPE_DIR = "/Users/elizabethkaplan/Desktop/phenotype"
OUT_DIR       = os.path.join(DERIV_ROOT, "ml")

CONDITIONS   = ["ec", "eo"]
PARAMS       = ["exponent", "offset"]
C_REG        = 0.1
N_PERM       = 500
RANDOM_STATE = 42
os.makedirs(OUT_DIR, exist_ok=True)
rng = np.random.RandomState(RANDOM_STATE)


# ============================================================
# fold-wise confound regression (confounds = LAST n columns of X)
# ============================================================
class ConfoundRegressor(BaseEstimator, TransformerMixin):
    """Regress the confound columns OUT of every signal column; return residuals.
    Fit on training data only, so it's leakage-safe inside cross-validation."""
    def __init__(self, n_confounds):
        self.n_confounds = n_confounds

    def fit(self, X, y=None):
        Xs, C = X[:, :-self.n_confounds], X[:, -self.n_confounds:]
        self.c_mean_ = C.mean(0)
        design = np.column_stack([np.ones(len(C)), C - self.c_mean_])
        self.beta_ = np.linalg.lstsq(design, Xs, rcond=None)[0]
        return self

    def transform(self, X):
        Xs, C = X[:, :-self.n_confounds], X[:, -self.n_confounds:]
        design = np.column_stack([np.ones(len(C)), C - self.c_mean_])
        return Xs - design @ self.beta_


def logistic():
    return LogisticRegression(penalty="l2", C=C_REG, max_iter=2000, solver="liblinear")

def loso_auc(pipe, X, y):
    proba = cross_val_predict(pipe, X, y, cv=LeaveOneOut(), method="predict_proba")[:, 1]
    return roc_auc_score(y, proba), balanced_accuracy_score(y, (proba > .5).astype(int))


# ============================================================
# BUILD FEATURES + COVARIATES + TARGET
# ============================================================
ap = pd.read_csv(APERIODIC_CSV, dtype={"subject": str})
ap = ap[ap["acq"].isin(CONDITIONS)]
wide = ap.pivot_table(index="subject", columns=["acq", "channel"], values=PARAMS)
wide.columns = [f"{p}_{a}_{c}" for (p, a, c) in wide.columns]
wide = wide.sort_index()
subjects = wide.index

parts = pd.read_csv(PARTICIPANTS, sep="\t")
parts["subject"] = parts["participant_id"].str.replace("sub-", "", regex=False)
parts = parts.set_index("subject")

life = pd.read_csv(os.path.join(PHENOTYPE_DIR, "lifestyle.tsv"), sep="\t")
life["subject"] = life["participant_id"].str.replace("sub-", "", regex=False)
life = life.set_index("subject")

y = (parts.loc[subjects, "group"] == 1).astype(int).values
Z = np.column_stack([
    parts.loc[subjects, "age"].astype(float).values,
    (life.loc[subjects, "daily_nicotine"] == 1).astype(float).values,   # 1 = daily nicotine
    life.loc[subjects, "alcohol_units"].astype(float).values,
])
X_eeg = wide.values
feat_names = list(wide.columns)
n_conf = Z.shape[1]

# ============================================================
# FOUR MODELS
# ============================================================
base = lambda: Pipeline([("imp", SimpleImputer()), ("sc", StandardScaler()), ("lr", logistic())])
resid = lambda: Pipeline([("imp", SimpleImputer()), ("cr", ConfoundRegressor(n_conf)),
                          ("sc", StandardScaler()), ("lr", logistic())])

auc_cov,  bacc_cov  = loso_auc(base(),  Z, y)
auc_eeg,  bacc_eeg  = loso_auc(base(),  X_eeg, y)
auc_both, bacc_both = loso_auc(base(),  np.hstack([X_eeg, Z]), y)
auc_res,  bacc_res  = loso_auc(resid(), np.hstack([X_eeg, Z]), y)

# permutation test on the confound-controlled model
null = np.empty(N_PERM)
Xrz = np.hstack([X_eeg, Z])
for i in range(N_PERM):
    yp = rng.permutation(y)
    pr = cross_val_predict(resid(), Xrz, yp, cv=LeaveOneOut(), method="predict_proba")[:, 1]
    null[i] = roc_auc_score(yp, pr)
p_res = (1 + np.sum(null >= auc_res)) / (N_PERM + 1)

print(f"covariates only        AUC {auc_cov:.3f}")
print(f"EEG only               AUC {auc_eeg:.3f}")
print(f"EEG + covariates       AUC {auc_both:.3f}")
print(f"EEG | covariates (resid) AUC {auc_res:.3f}  perm p {p_res:.4f}")

# ============================================================
# COEFFICIENT TOPOMAP (EEG-only model, refit on all data)
# ============================================================
pipe = base(); pipe.fit(X_eeg, y)
coef_by_feat = dict(zip(feat_names, pipe.named_steps["lr"].coef_.ravel()))
topo_note = "topomap skipped"
try:
    import mne
    channels = sorted(ap["channel"].unique())
    info = mne.create_info(channels, sfreq=1.0, ch_types="eeg")
    info.set_montage(mne.channels.make_standard_montage("standard_1005"), on_missing="ignore")
    panels = [(p, c) for p in PARAMS for c in CONDITIONS]
    fig, axes = plt.subplots(1, len(panels), figsize=(4 * len(panels), 4))
    axes = np.atleast_1d(axes)
    for ax, (param, acq) in zip(axes, panels):
        vals = np.array([coef_by_feat.get(f"{param}_{acq}_{ch}", np.nan) for ch in channels])
        vmax = np.nanmax(np.abs(vals))
        im, _ = mne.viz.plot_topomap(vals, info, axes=ax, show=False, cmap="RdBu_r",
                                     vlim=(-vmax, vmax), contours=0)
        ax.set_title(f"{param} · {acq}")
    fig.suptitle("Logistic coefficients (red = predicts current HC use)")
    fig.colorbar(im, ax=list(axes), shrink=.6, label="coef")
    fig.savefig(os.path.join(OUT_DIR, "hc_coef_topomap.png"), dpi=140, bbox_inches="tight")
    plt.close(fig)
    topo_note = "saved hc_coef_topomap.png"
except Exception as e:
    topo_note = f"topomap skipped ({type(e).__name__}: {e})"
print(topo_note)

# ============================================================
# SAVE RESULTS
# ============================================================
with open(os.path.join(OUT_DIR, "hc_covariate_control_results.txt"), "w") as f:
    f.write("HC-use decoding — confound control (age, nicotine, alcohol)\n" + "=" * 58 + "\n")
    f.write(f"n subjects = {len(subjects)} | current={int(y.sum())} non-current={int((1-y).sum())}\n")
    f.write(f"covariates = age, daily_nicotine, alcohol_units\n")
    f.write(f"CV = leave-one-subject-out | model = L2 logistic (C={C_REG})\n\n")
    f.write(f"{'model':<28}{'AUC':>7}{'bal.acc':>9}\n" + "-" * 44 + "\n")
    f.write(f"{'covariates only':<28}{auc_cov:>7.3f}{bacc_cov:>9.3f}\n")
    f.write(f"{'EEG only':<28}{auc_eeg:>7.3f}{bacc_eeg:>9.3f}\n")
    f.write(f"{'EEG + covariates':<28}{auc_both:>7.3f}{bacc_both:>9.3f}\n")
    f.write(f"{'EEG | covariates (resid)':<28}{auc_res:>7.3f}{bacc_res:>9.3f}\n\n")
    f.write(f"confound-controlled permutation p = {p_res:.4f} "
            f"({N_PERM} perms, null AUC mean {null.mean():.3f})\n\n")
    f.write("interpretation: if 'covariates only' is ~0.5 and 'EEG | covariates' stays "
            "well above chance, the HC decode is not explained by age/nicotine/alcohol.\n\n")
    f.write(topo_note + "\n")
print("Saved ->", OUT_DIR)

covariates only        AUC 0.542
EEG only               AUC 0.680
EEG + covariates       AUC 0.670
EEG | covariates (resid) AUC 0.658  perm p 0.0479
saved hc_coef_topomap.png
Saved -> /Users/elizabethkaplan/Desktop/ds007615/ds007615/derivatives/preproc/ml
